# Corporación Favorita Grocery Sales Forecasting


<img src='https://img.cnbce.com/2/740/415/storage/files/images/2025/01/02/enflasyon-market-u9tr.jpg'>


Bu çalışmada `Corporación Favorita Grocery Sales Forecasting` yarışması için mağaza, ürün ve promosyon bilgileri kullanılarak satış tahmini yapan bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Veri birleştirme ve ön işleme
5. Özellik çıkarımı
6. Model kurma
7. RMSE değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
!pip install catboost py7zr -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.3/494.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 7.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os
import io
import zipfile
import tempfile
import py7zr

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [3]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/favorita-grocery-sales-forecasting.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError('Zip dosyası bulunamadı. Colab Data Dosyaları içindeki gerçek dosya adını kontrol et.')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

def find_zip_member(members, candidates):
    for candidate in candidates:
        for member in members:
            if member.endswith(candidate):
                return member
    raise FileNotFoundError(f'{candidates[0]} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, ['train.csv', 'train.csv.zip', 'train.csv.7z'])
test_member = find_zip_member(zip_members, ['test.csv', 'test.csv.zip', 'test.csv.7z'])
stores_member = find_zip_member(zip_members, ['stores.csv', 'stores.csv.zip', 'stores.csv.7z'])
items_member = find_zip_member(zip_members, ['items.csv', 'items.csv.zip', 'items.csv.7z'])
oil_member = find_zip_member(zip_members, ['oil.csv', 'oil.csv.zip', 'oil.csv.7z'])
holidays_member = find_zip_member(zip_members, ['holidays_events.csv', 'holidays_events.csv.zip', 'holidays_events.csv.7z'])
sample_submission_member = find_zip_member(zip_members, ['sample_submission.csv', 'sample_submission.csv.zip', 'sample_submission.csv.7z'])


Mounted at /content/drive


## 3. Veri Dosyalarını Yükleme


In [4]:
sample_train_rows = 800000
sample_test_rows = 200000

def read_csv_from_zip(zip_path, member_name, nrows=None):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            if member_name.endswith('.zip'):
                inner_bytes = f.read()
                with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as inner_zip:
                    inner_name = inner_zip.namelist()[0]
                    with inner_zip.open(inner_name) as inner_f:
                        return pd.read_csv(inner_f, nrows=nrows)
            if member_name.endswith('.7z'):
                inner_bytes = f.read()
                with tempfile.TemporaryDirectory() as tmpdir:
                    archive_path = os.path.join(tmpdir, os.path.basename(member_name))
                    with open(archive_path, 'wb') as archive_file:
                        archive_file.write(inner_bytes)
                    with py7zr.SevenZipFile(archive_path, mode='r') as archive:
                        archive.extractall(path=tmpdir)
                    extracted_files = []
                    for root, _, files in os.walk(tmpdir):
                        for file_name in files:
                            if file_name.endswith('.csv'):
                                extracted_files.append(os.path.join(root, file_name))
                    if not extracted_files:
                        raise FileNotFoundError(f'{member_name} içinden csv çıkarılamadı.')
                    return pd.read_csv(extracted_files[0], nrows=nrows)
            return pd.read_csv(f, nrows=nrows)

train = read_csv_from_zip(zip_path, train_member, nrows=sample_train_rows)
test = read_csv_from_zip(zip_path, test_member, nrows=sample_test_rows)
stores = read_csv_from_zip(zip_path, stores_member)
items = read_csv_from_zip(zip_path, items_member)
oil = read_csv_from_zip(zip_path, oil_member)
holidays = read_csv_from_zip(zip_path, holidays_member)
sample_submission = read_csv_from_zip(zip_path, sample_submission_member)

print('Train shape (sample):', train.shape)
print('Test shape (sample):', test.shape)
print('Stores shape:', stores.shape)
print('Items shape:', items.shape)
print('Oil shape:', oil.shape)
print('Holidays shape:', holidays.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape (sample): (800000, 6)
Test shape (sample): (200000, 5)
Stores shape: (54, 5)
Items shape: (4100, 4)
Oil shape: (1218, 2)
Holidays shape: (350, 6)
Sample submission shape: (3370464, 2)


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


## 4. Veri Birleştirme ve Ön İşleme


In [5]:
train_df = train.merge(stores, on='store_nbr', how='left').merge(items, on='item_nbr', how='left')
test_df = test.merge(stores, on='store_nbr', how='left').merge(items, on='item_nbr', how='left')

oil['date'] = pd.to_datetime(oil['date'])
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

train_df = train_df.merge(oil, on='date', how='left')
test_df = test_df.merge(oil, on='date', how='left')

print('Train merged shape:', train_df.shape)
print('Test merged shape:', test_df.shape)
train_df.head()


Train merged shape: (800000, 14)
Test merged shape: (200000, 13)


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,family,class,perishable,dcoilwtico
0,0,2013-01-01,25,103665,7.0,NaN,Salinas,Santa Elena,D,1,BREAD/BAKERY,2712,1,NaN
1,1,2013-01-01,25,105574,1.0,NaN,Salinas,Santa Elena,D,1,GROCERY I,1045,0,NaN
2,2,2013-01-01,25,105575,2.0,NaN,Salinas,Santa Elena,D,1,GROCERY I,1045,0,NaN
3,3,2013-01-01,25,108079,1.0,NaN,Salinas,Santa Elena,D,1,GROCERY I,1030,0,NaN
4,4,2013-01-01,25,108701,1.0,NaN,Salinas,Santa Elena,D,1,DELI,2644,1,NaN


## 5. Özellik Çıkarımı


In [6]:
for df in [train_df, test_df]:
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

feature_columns = [
    'store_nbr', 'item_nbr', 'onpromotion', 'city', 'state', 'type', 'cluster',
    'family', 'class', 'perishable', 'dcoilwtico', 'year', 'month', 'day', 'dayofweek', 'is_weekend'
]

x = train_df[feature_columns].copy()
y = np.log1p(train_df['unit_sales'].clip(lower=0))
test_x = test_df[feature_columns].copy()

categorical_features = ['store_nbr', 'item_nbr', 'city', 'state', 'type', 'family']
for col in categorical_features:
    x[col] = x[col].astype(str)
    test_x[col] = test_x[col].astype(str)

x_train, x_valid, y_train, y_valid = train_test_split(
    x, y, test_size=0.2, random_state=42
)

cat_feature_indices = [x.columns.get_loc(col) for col in categorical_features]

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (640000, 16)
x_valid shape: (160000, 16)


## 6. Model Kurma


In [7]:
# Bu bölümde mağaza ve ürün özellikleri ile satış tahmini yapan başlangıç modelini kuruyoruz.


In [8]:
model = CatBoostRegressor(
    loss_function='RMSE',
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=0
)

model.fit(x_train, y_train, cat_features=cat_feature_indices)


CatBoostRegressor(depth=8, iterations=300, learning_rate=0.1, loss_function='RMSE', random_state=42, verbose=0)

## 7. RMSE Değerlendirmesi


In [9]:
# Bu bölümde modelin hata düzeyini log dönüşümlü hedef üzerinde RMSE ile ölçüyoruz.


In [10]:
valid_preds = model.predict(x_valid)
rmse = root_mean_squared_error(y_valid, valid_preds)
print('Validation RMSE:', round(rmse, 5))


Validation RMSE: 0.51021


## 8. Test Tahmini ve Submission


In [11]:
# Bu bölümde test verisi için satış tahminlerini üretip submission dosyasını oluşturuyoruz.


In [12]:
test_preds = model.predict(test_x)
final_preds = np.expm1(test_preds)
final_preds = np.clip(final_preds, a_min=0, a_max=None)

submission = pd.DataFrame({
    'id': test_df['id'].values,
    'unit_sales': final_preds
})

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (200000, 2)


,id,unit_sales
0,125497040,1.355870
1,125497041,1.688444
2,125497042,1.715702
3,125497043,2.068320
4,125497044,2.911970


In [13]:
output_path = '/content/favorita_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/favorita_submission.csv


## 9. Sonuç


Bu çalışmada Corporación Favorita Grocery Sales Forecasting yarışması için mağaza, ürün ve promosyon bilgileri kullanılarak satış tahmini yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 0.51021 RMSE değeri elde etti ve test verisi için satış tahminleri üretildi.
